# Week 8 — Agentic AI: Single-Agent Task Routing Pipeline

## 1. Baseline structural mechanics — the Calculator tool

Pulls a math expression out of the query (after stripping the `"calculate"`
trigger word), validates it only contains safe characters, then evaluates it.

In [1]:
import re
import json

_ALLOWED_CHARS = re.compile(r"^[0-9\.\+\-\*\/\(\)\s%]+$")


def _extract_expression(query: str) -> str:
    expression = re.sub(r"calculate", "", query, flags=re.IGNORECASE)
    expression = expression.replace(":", "").strip()
    return expression


def calculator(query: str) -> dict:
    """Evaluates a math expression found inside the query string."""
    expression = _extract_expression(query)

    if not expression:
        return {"type": "error", "result": "No mathematical expression found in query."}

    if not _ALLOWED_CHARS.match(expression):
        return {"type": "error", "result": f"Invalid characters in expression: \'{expression}\'"}

    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return {"type": "calculation", "result": result}
    except ZeroDivisionError:
        return {"type": "error", "result": "Division by zero."}
    except Exception as e:
        return {"type": "error", "result": f"Could not evaluate expression: {e}"}


# Quick sanity checks
print(calculator("calculate 5 + 3 * 2"))
print(calculator("calculate 10 / 0"))
print(calculator("calculate abc + def"))

{'type': 'calculation', 'result': 11}
{'type': 'error', 'result': 'Division by zero.'}
{'type': 'error', 'result': "Invalid characters in expression: 'abc + def'"}


## 2. Baseline structural mechanics — the Keyword Extractor tool

Lowercases the text, strips punctuation, removes stopwords/the trigger word,
and returns the remaining significant words.

In [2]:
_STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "in", "on", "at", "to", "for", "of", "with", "and", "or", "but",
    "this", "that", "these", "those", "it", "its", "as", "by", "from",
    "keywords", "keyword", "extract", "find", "get", "please", "me",
    "i", "you", "we", "they", "he", "she", "them", "my", "your", "about"
}


def _clean_text(text: str) -> list:
    text = re.sub(r"[^\w\s]", "", text.lower())
    return text.split()


def extract_keywords(query: str) -> dict:
    """Extracts keywords from a query string."""
    words = _clean_text(query)
    keywords = [w for w in words if w not in _STOPWORDS and len(w) > 2]

    seen = set()
    unique_keywords = []
    for w in keywords:
        if w not in seen:
            seen.add(w)
            unique_keywords.append(w)

    if not unique_keywords:
        return {"type": "error", "result": "No keywords found in query."}

    return {"type": "keywords", "result": unique_keywords}


# Quick sanity checks
print(extract_keywords("extract keywords from this sentence about machine learning models"))
print(extract_keywords("keywords"))

{'type': 'keywords', 'result': ['sentence', 'machine', 'learning', 'models']}
{'type': 'error', 'result': 'No keywords found in query.'}


## 3. The `agent(query)` routing pipeline



In [3]:
def _general_response(query: str) -> dict:
    return {
        "type": "general",
        "result": f"I received your message: \'{query}\'. "
                   f"I don\'t have a specific tool for this, so this is a general response."
    }


def agent(query: str) -> dict:
    """Routes a query to the correct tool and returns a structured JSON-style dict."""
    if not isinstance(query, str) or not query.strip():
        return {"type": "error", "result": "Query must be a non-empty string."}

    query_lower = query.lower()

    try:
        if "calculate" in query_lower:
            return calculator(query)
        elif "keywords" in query_lower:
            return extract_keywords(query)
        else:
            return _general_response(query)
    except Exception as e:
        return {"type": "error", "result": f"Unexpected agent failure: {e}"}


# Single example call
print(json.dumps(agent("calculate 45 * 2 - 10"), indent=2))

{
  "type": "calculation",
  "result": 80
}


## 4. Automated validation checks

Running a fixed array of test queries through the agent to confirm routing,
JSON formatting, and error handling all behave correctly.

In [4]:
test_queries = [
    "calculate 12 * 4 - 5",
    "please extract keywords from this paragraph about neural networks and gradient descent",
    "hello, how are you today?",
    "calculate 10 / 0",
    "",
    "calculate abc + def",
    "keywords",
]

for q in test_queries:
    response = agent(q)
    assert isinstance(response, dict)
    assert "type" in response and "result" in response
    print(f"Query: {q!r}")
    print(f"  -> {json.dumps(response)}\n")

print("All validation checks passed: every response is a dict with \'type\' and \'result\' keys.")

Query: 'calculate 12 * 4 - 5'
  -> {"type": "calculation", "result": 43}

Query: 'please extract keywords from this paragraph about neural networks and gradient descent'
  -> {"type": "keywords", "result": ["paragraph", "neural", "networks", "gradient", "descent"]}

Query: 'hello, how are you today?'
  -> {"type": "general", "result": "I received your message: 'hello, how are you today?'. I don't have a specific tool for this, so this is a general response."}

Query: 'calculate 10 / 0'
  -> {"type": "error", "result": "Division by zero."}

Query: ''
  -> {"type": "error", "result": "Query must be a non-empty string."}

Query: 'calculate abc + def'
  -> {"type": "error", "result": "Invalid characters in expression: 'abc + def'"}

Query: 'keywords'
  -> {"type": "error", "result": "No keywords found in query."}

All validation checks passed: every response is a dict with 'type' and 'result' keys.


## 5. Interactive verification loop


In [5]:
while True:
    user_query = input("Query (or \'exit\' to quit): ")
    if user_query.strip().lower() == "exit":
        print("Exiting interactive loop.")
        break
    print(json.dumps(agent(user_query), indent=2))

{
  "type": "general",
  "result": "I received your message: '3+5'. I don't have a specific tool for this, so this is a general response."
}
{
  "type": "calculation",
  "result": 110
}
{
  "type": "calculation",
  "result": 8
}
{
  "type": "calculation",
  "result": 60
}
{
  "type": "general",
  "result": "I received your message: 'what is the weather today?'. I don't have a specific tool for this, so this is a general response."
}
Exiting interactive loop.


## 6. Observations

- Routing just checks if "calculate" or "keywords" shows up in the query. Anything else falls back to a general response, so nothing goes unhandled.
- Didn't want to use eval() directly on raw input since that's risky, so I check the expression only has numbers/math symbols first. Bad input returns an error instead of crashing.
- Every response type (calculation, keywords, general, error) returns the same "type" and "result" fields, so the output format stays consistent no matter what.
- Ran into a bug where "Calculate 5+5" with a capital C wasn't getting routed properly. Fixed it by lowercasing the query before checking for keywords.